# # AI Guardrail - Dataset Cleaning & Model Training Log

## What this stage is for

Our third layer of defense will be a machine learning model that can catch sneaky or disguised attacks that a simple word-matching filter cannot. Before we can train that model, we need a clean, well-balanced dataset of real examples to learn from. This log covers everything we did to gather, clean, and prepare that data.

## Gathering our data from multiple sources

Instead of relying on a single dataset, we combined four different sources, since each one brings something different to the table:

- **Our original JSONL dataset** - 500 example prompts, labeled malicious or benign
- **Our original CSV dataset** - 686 example prompts, with extra detail like language, category, and complexity
- **A jailbreak prompt dataset** - 1,405 real jailbreak attempts, collected from Reddit, Discord, and other online communities as part of a published academic security research project
- **A matching "regular" prompt dataset** — 5,721 normal, non-attack messages from the same sources, giving our model real examples of what an ordinary message looks like

Using multiple sources like this gives our model a wider, more realistic variety of both attacks and normal conversation to learn from, rather than learning only the style of a single dataset.

In [23]:
import pandas as pd
import json

# Load datasets (jailbreak + regular)
df_jailbreak = pd.read_csv("data/jailbreak_prompts.csv")
df_jailbreak['label'] = 'malicious'

df_regular = pd.read_csv("data/regular_prompts.csv")
df_regular['label'] = 'benign'

#Load the two datasets
jsonl_rows = []
with open("data/Prompt_INJECTION_And_Benign_DATASET.jsonl") as f:
    for line in f:
        jsonl_rows.append(json.loads(line))
df_jsonl = pd.DataFrame(jsonl_rows)

df_csv = pd.read_csv("data/prompt_injection_detection_dataset.csv")

#Inspect all four before merging anything
print("Jailbreak shape:", df_jailbreak.shape)
print("Jailbreak columns:", df_jailbreak.columns.tolist())
print()
print("Regular shape:", df_regular.shape)
print("Regular columns:", df_regular.columns.tolist())
print()
print("JSONL shape:", df_jsonl.shape)
print("JSONL columns:", df_jsonl.columns.tolist())
print()
print("CSV shape:", df_csv.shape)
print("CSV columns:", df_csv.columns.tolist())

Jailbreak shape: (1405, 10)
Jailbreak columns: ['platform', 'source', 'prompt', 'jailbreak', 'created_at', 'date', 'community', 'community_id', 'previous_community_id', 'label']

Regular shape: (5721, 7)
Regular columns: ['platform', 'source', 'prompt', 'jailbreak', 'created_at', 'date', 'label']

JSONL shape: (500, 6)
JSONL columns: ['id', 'prompt', 'label', 'attack_type', 'context', 'response']

CSV shape: (686, 10)
CSV columns: ['id', 'text', 'label', 'category', 'subcategory', 'language', 'complexity', 'target_goal', 'source', 'split']


## Standardizing the data

Each of our four sources used different column names and different wording for their labels. Before we could combine them, we needed everything to match:

- We renamed each dataset's text column to a single consistent name, `text`
- We renamed each dataset's label column to a single consistent name, `label`
- We converted differing label words (for example, "injection") so that every source consistently uses either `malicious` or `benign`

In [24]:
# Step 2: Standardize each dataset down to just 'text' and 'label'

# Jailbreak (malicious) - text column is 'prompt'
df_jailbreak_clean = df_jailbreak[['prompt', 'label']].rename(columns={'prompt': 'text'})

# Regular (benign) - text column is 'prompt'
df_regular_clean = df_regular[['prompt', 'label']].rename(columns={'prompt': 'text'})

# Your original JSONL - text column is already 'prompt'
df_jsonl_clean = df_jsonl[['prompt', 'label']].rename(columns={'prompt': 'text'})

# Your original CSV - text column is already 'text', but labels say 'injection' not 'malicious'
df_csv_clean = df_csv[['text', 'label']].copy()
df_csv_clean['label'] = df_csv_clean['label'].replace({'injection': 'malicious'})

# Step 3: Normalize AI product names (so the model doesn't cheat by spotting brand names)
import re

def normalize_ai_names(text):
    if not isinstance(text, str):
        return text
    text = re.sub(r'\bchatgpt\b', '[AI]', text, flags=re.IGNORECASE)
    text = re.sub(r'\bgpt-?[0-9]?\b', '[AI]', text, flags=re.IGNORECASE)
    text = re.sub(r'\bclaude\b', '[AI]', text, flags=re.IGNORECASE)
    text = re.sub(r'\bgemini\b', '[AI]', text, flags=re.IGNORECASE)
    text = re.sub(r'\bllama ?[0-9]?\b', '[AI]', text, flags=re.IGNORECASE)
    return text

df_jailbreak_clean['text'] = df_jailbreak_clean['text'].apply(normalize_ai_names)
df_regular_clean['text'] = df_regular_clean['text'].apply(normalize_ai_names)

# Confirm all four now have identical structure
for name, df in [("Jailbreak", df_jailbreak_clean), ("Regular", df_regular_clean),
                  ("JSONL", df_jsonl_clean), ("CSV", df_csv_clean)]:
    print(f"{name}: {df.shape}, labels: {df['label'].unique()}")

Jailbreak: (1405, 2), labels: ['malicious']
Regular: (5721, 2), labels: ['benign']
JSONL: (500, 2), labels: ['malicious' 'benign']
CSV: (686, 2), labels: ['benign' 'malicious']


## Removing AI brand names from the text

Many of the jailbreak examples specifically mention "ChatGPT" by name. Rather than deleting the word outright (which can leave sentences reading awkwardly), we replaced any AI product name (ChatGPT, GPT, Claude, Gemini, Llama) with a neutral placeholder, `[AI]`.

We did this deliberately, not just for tidiness: if our malicious examples frequently mention a specific AI brand name and our benign examples don't, our model could quietly learn to treat "mentions ChatGPT" as a shortcut for "this is an attack" — which is not actually what makes something an attack. Removing the brand names forces the model to learn the real structure and intent behind an attack instead.

## Combining everything and removing duplicates

Once all four sources were standardized, we combined them into a single dataset and removed any exact duplicate messages that appeared more than once across sources.

- Before removing duplicates: **8,312 examples**
- After removing duplicates: **8,033 examples**

In [25]:
# Step 4: Combine all four into one dataset
df_merged = pd.concat(
    [df_jailbreak_clean, df_regular_clean, df_jsonl_clean, df_csv_clean],
    ignore_index=True
)

print("Before removing duplicates:", df_merged.shape)

# Remove exact duplicate text entries (same sentence appearing more than once)
df_merged = df_merged.drop_duplicates(subset='text')

print("After removing duplicates:", df_merged.shape)
print()
print("Final label balance:")
print(df_merged['label'].value_counts())

Before removing duplicates: (8312, 2)
After removing duplicates: (8033, 2)

Final label balance:
label
benign       6182
malicious    1851
Name: count, dtype: int64


## Save merged dataset  
This gives us one clean, permanent file to train from instead of rerunning the merge everytime

In [26]:
df_merged.to_csv("data/merged_dataset.csv", index=False)
print("Saved to data/merged_dataset.csv")

Saved to data/merged_dataset.csv


## Balancing the dataset

To fix this, we reduced the benign examples down to match the number of malicious examples, giving the model a fair, roughly equal mix of both to learn from during training. This does mean we are setting aside a large portion of benign examples we won't use, which is a deliberate trade-off: we are prioritizing a fair, honest model over using every last row of data we collected.

In [27]:
# Balance the dataset
malicious_count = df_merged['label'].value_counts()['malicious']

df_benign = df_merged[df_merged['label'] == 'benign'].sample(n=malicious_count, random_state=42)
df_malicious = df_merged[df_merged['label'] == 'malicious']

df_balanced = pd.concat([df_benign, df_malicious], ignore_index=True)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print("Balanced dataset shape:", df_balanced.shape)
print(df_balanced['label'].value_counts())

Balanced dataset shape: (3702, 2)
label
benign       1851
malicious    1851
Name: count, dtype: int64


## Splitting the data and converting text to numbers

Before training, we split our balanced dataset into two parts: a training set the model learns from, and a test set we hold back to check how well it performs on examples it has never seen. We use an 80/20 split — 80% for training, 20% for testing.

Machine learning models cannot work with raw text directly — they need numbers. We use a technique called TF-IDF (Term Frequency-Inverse Document Frequency) to convert each message into a row of numbers representing which words appear and how significant each word is. Common words that appear everywhere (like "the" or "is") get a low weight, while distinctive words that help tell attacks apart from normal messages get a higher weight.

In [28]:
%pip install scikit-learn


/home/Student/ai-guardrail/env/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [29]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Split into features (X) and labels (y)
X = df_balanced['text']
y = df_balanced['label']

# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training examples:", X_train.shape[0])
print("Testing examples:", X_test.shape[0])

# Convert text into numbers using TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Training matrix shape:", X_train_vec.shape)
print("Testing matrix shape:", X_test_vec.shape)

Training examples: 2961
Testing examples: 741
Training matrix shape: (2961, 5000)
Testing matrix shape: (741, 5000)


In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_vec, y_train)

print("Model trained.")

Model trained.


In [33]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test_vec)

print(classification_report(y_test, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred, labels=['benign', 'malicious']))

              precision    recall  f1-score   support

      benign       0.88      0.87      0.87       371
   malicious       0.87      0.88      0.87       370

    accuracy                           0.87       741
   macro avg       0.87      0.87      0.87       741
weighted avg       0.87      0.87      0.87       741

Confusion matrix:
[[322  49]
 [ 46 324]]


In [34]:
import joblib

joblib.dump(model, "models/classifier.pkl")
joblib.dump(vectorizer, "models/vectorizer.pkl")

print("Model and vectorizer saved to models/")

Model and vectorizer saved to models/
